# Salary Prediction Using Linear Regression

**Objective:** Predict IT salary rates based on job title, experience level, employment type, company size, location, and remote work ratio.

**Dataset:** salaries.csv containing salary information for IT professionals

**Method:** Linear Regression

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
sns.set_palette('husl')

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('salaries.csv')

# Display basic information
print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Data types and info
print('Dataset Info:')
df.info()

## 3. Exploratory Data Analysis (EDA)

Analyze the data to understand patterns and identify features that correlate with salary.

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Statistical summary
print('Statistical Summary:')
df.describe()

In [ ]:
# Salary distribution
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(df['salary_in_usd'], bins=50, edgecolor='black')
plt.xlabel('Salary (USD)')
plt.ylabel('Frequency')
plt.title('Salary Distribution')

plt.subplot(1, 2, 2)
plt.boxplot(df['salary_in_usd'])
plt.ylabel('Salary (USD)')
plt.title('Salary Boxplot')

plt.tight_layout()
plt.show()

print(f'Salary Statistics:')
print(f'Mean: ${df["salary_in_usd"].mean():,.2f}')
print(f'Median: ${df["salary_in_usd"].median():,.2f}')
print(f'Min: ${df["salary_in_usd"].min():,.2f}')
print(f'Max: ${df["salary_in_usd"].max():,.2f}')

In [ ]:
# Experience level vs Salary - shows correlation
plt.figure(figsize=(10, 6))
df.boxplot(column='salary_in_usd', by='experience_level', figsize=(10, 6))
plt.xlabel('Experience Level')
plt.ylabel('Salary (USD)')
plt.title('Salary by Experience Level')
plt.suptitle('')
plt.show()

print('\nAverage Salary by Experience Level:')
print(df.groupby('experience_level')['salary_in_usd'].mean().sort_values(ascending=False))

In [ ]:
# Company size vs Salary
plt.figure(figsize=(10, 6))
df.boxplot(column='salary_in_usd', by='company_size', figsize=(10, 6))
plt.xlabel('Company Size')
plt.ylabel('Salary (USD)')
plt.title('Salary by Company Size')
plt.suptitle('')
plt.show()

print('\nAverage Salary by Company Size:')
print(df.groupby('company_size')['salary_in_usd'].mean().sort_values(ascending=False))

In [ ]:
# Remote ratio vs Salary
plt.figure(figsize=(10, 6))
df.boxplot(column='salary_in_usd', by='remote_ratio', figsize=(10, 6))
plt.xlabel('Remote Ratio (%)')
plt.ylabel('Salary (USD)')
plt.title('Salary by Remote Work Ratio')
plt.suptitle('')
plt.show()

print('\nAverage Salary by Remote Ratio:')
print(df.groupby('remote_ratio')['salary_in_usd'].mean().sort_values(ascending=False))

In [ ]:
# Analyze categorical variables
print('Employment Type Distribution:')
print(df['employment_type'].value_counts())
print('\nTop 10 Job Titles:')
print(df['job_title'].value_counts().head(10))
print(f'\nTotal unique job titles: {df["job_title"].nunique()}')

## 4. Data Preprocessing

Group similar job titles into categories to reduce dimensionality and improve model performance.

In [ ]:
# Function to group job titles into categories
def group_job_title(title):
    title_lower = title.lower()
    if 'data scientist' in title_lower or 'data science' in title_lower:
        return 'Data Scientist'
    elif 'engineer' in title_lower:
        return 'Engineer'
    elif 'manager' in title_lower or 'director' in title_lower or 'head' in title_lower or 'lead' in title_lower:
        return 'Manager/Leadership'
    elif 'analyst' in title_lower or 'analytics' in title_lower:
        return 'Analyst'
    elif 'ml' in title_lower or 'machine learning' in title_lower or 'ai' in title_lower or 'artificial intelligence' in title_lower:
        return 'ML/AI Specialist'
    else:
        return 'Other'

# Apply grouping
df['job_category'] = df['job_title'].apply(group_job_title)

print('Job Category Distribution:')
print(df['job_category'].value_counts())
print('\nAverage Salary by Job Category:')
print(df.groupby('job_category')['salary_in_usd'].mean().sort_values(ascending=False))

## 5. Feature Encoding

Convert categorical variables to numerical format for the linear regression model.

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Label encode experience_level (ordinal: EN < MI < SE < EX)
experience_mapping = {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}
df_processed['experience_level_encoded'] = df_processed['experience_level'].map(experience_mapping)

print('Experience Level Encoding:')
print(df_processed[['experience_level', 'experience_level_encoded']].drop_duplicates().sort_values('experience_level_encoded'))

In [ ]:
# One-hot encode categorical variables
categorical_features = ['employment_type', 'job_category', 'company_size']

df_encoded = pd.get_dummies(df_processed, columns=categorical_features, drop_first=True)

print(f'Original shape: {df_processed.shape}')
print(f'Encoded shape: {df_encoded.shape}')
print(f'\nNew columns after encoding: {df_encoded.shape[1] - df_processed.shape[1]} added')

## 6. Model Training

Split the data and train a linear regression model to predict salary.

In [ ]:
# Select features for the model
feature_columns = ['work_year', 'experience_level_encoded', 'remote_ratio'] + \
                  [col for col in df_encoded.columns if col.startswith(('employment_type_', 'job_category_', 'company_size_'))]

X = df_encoded[feature_columns]
y = df_encoded['salary_in_usd']

print(f'Features selected: {len(feature_columns)}')
print(f'Feature names: {feature_columns[:10]}...')  # Show first 10
print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Testing set size: {X_test.shape[0]} samples')
print(f'\nTraining set: {X_train.shape[0]/len(X)*100:.1f}%')
print(f'Testing set: {X_test.shape[0]/len(X)*100:.1f}%')

In [ ]:
# Create and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print('Linear Regression Model Trained Successfully!')
print(f'\nModel Intercept: ${model.intercept_:,.2f}')
print(f'\nTop 10 Feature Coefficients:')
coef_df = pd.DataFrame({'Feature': feature_columns, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)
print(coef_df.head(10))

## 7. Model Evaluation

Evaluate the model's performance using various metrics and visualizations.

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print('Model Performance Metrics:')
print(f'R² Score: {r2:.4f}')
print(f'Mean Absolute Error (MAE): ${mae:,.2f}')
print(f'Mean Squared Error (MSE): ${mse:,.2f}')
print(f'Root Mean Squared Error (RMSE): ${rmse:,.2f}')
print(f'\nInterpretation: The model explains {r2*100:.2f}% of the variance in salary.')

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Salary (USD)')
plt.ylabel('Predicted Salary (USD)')
plt.title('Actual vs Predicted Salary')
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - y_pred

plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Salary (USD)')
plt.ylabel('Residuals (USD)')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

## 8. Making Predictions

Use the trained model to predict salary for new data.

In [ ]:
# Example: Predict salary for a new individual
# Senior Engineer, Full-time, Medium company, 100% remote, in 2025

# Create sample data matching our encoding
sample_data = pd.DataFrame({
    'work_year': [2025],
    'experience_level_encoded': [2],  # SE (Senior)
    'remote_ratio': [100]
})

# Add encoded features (set all to 0, then set the ones we want to 1)
for col in feature_columns:
    if col not in sample_data.columns:
        sample_data[col] = 0

# Set the specific categorical features
if 'employment_type_FT' in feature_columns:
    sample_data['employment_type_FT'] = 1
if 'job_category_Engineer' in feature_columns:
    sample_data['job_category_Engineer'] = 1
if 'company_size_M' in feature_columns:
    sample_data['company_size_M'] = 1

# Reorder columns to match training data
sample_data = sample_data[feature_columns]

# Make prediction
predicted_salary = model.predict(sample_data)[0]

print('Prediction Example:')
print('Profile: Senior Engineer, Full-time, Medium company, 100% remote, Year 2025')
print(f'Predicted Salary: ${predicted_salary:,.2f} USD')

## 9. Conclusion

**Key Findings:**
- Experience level shows strong correlation with salary (higher experience = higher salary)
- Job category (Manager/Leadership, Engineer, Data Scientist, etc.) significantly impacts salary
- Company size and remote work ratio also influence compensation
- The linear regression model successfully predicts IT salaries based on these factors

**Model Performance:**
- The R² score indicates how well the model explains salary variance
- RMSE shows the average prediction error in USD

**Next Steps:**
- Consider feature engineering (interactions, polynomial features)
- Try other regression models (Ridge, Lasso, Random Forest)
- Analyze location-specific salary patterns